# 02 — Vectorization Nodes (Open-Iris)

This notebook documents and demonstrates the vectorization stage of the Open-Iris pipeline.  
Focus nodes: ContouringAlgorithm, ContourInterpolation, ContourPointNoiseEyeballDistanceFilter, and Smoothing.

## Node: ContouringAlgorithm

### What it does
ContouringAlgorithm converts segmentation masks into vectorized contour representations. It extracts the boundaries of the pupil, iris, and eyeball by tracing their edges and returning them as polygon coordinate arrays. These contours are used in later stages for geometric refinement and normalization.

In other words it uses filter functions to clean up the contours (removing noise or bad shapes)

### Key Parameters

| Parameter | Default | What it controls |
|---|---|---|
| `contour_filters` | `[filter_polygon_areas]` | Filters out noisy or invalid contour shapes from the detected contours |

### Code Example

The following code initializes the ContouringAlgorithm node. This node is responsible for extracting contour boundaries from the segmentation masks of the pupil, iris, and eyeball.

In [ ]:
from iris.nodes.vectorization.contouring import ContouringAlgorithm

node = ContouringAlgorithm()
print(node)

### What to Look For

A successful run of the ContouringAlgorithm node should produce polygon contour representations of the pupil, iris, and eyeball regions. These contours are returned as arrays of coordinate points outlining the boundaries of each region.

Good output will contain smooth, continuous contours that closely follow the edges of the segmented regions. Poor output may contain fragmented contours, missing boundaries, or irregular shapes caused by noise in the segmentation mask.

## Node: ContourInterpolation

### What it does

ContourInterpolation refines the raw contour points produced during the contouring stage. It fills in gaps between boundary points so that the iris, pupil, and eyeball outlines become smoother and more continuous.

### Key Parameters

| Parameter | Default | What it controls |
|---|---|---|
| max_distance_between_boundary_points | 0.01 | Maximum allowed distance between contour boundary points. If points are farther apart than this fraction of the iris diameter, interpolation fills in additional points to smooth the contour. |

### Code Example

The following code initializes the ContourInterpolation node. This node is used to fill in gaps between contour boundary points so the resulting outlines are smoother and more continuous.

In [ ]:
from iris.nodes.geometry_refinement.contour_interpolation import ContourInterpolation

interp_node = ContourInterpolation()
print(interp_node)

### What to Look For

A successful run of the ContourInterpolation node should help create smoother and more continuous contour boundaries by filling in gaps between existing contour points.

Good output will show boundary curves with fewer breaks and more evenly spaced points. Poor output may still contain gaps, jagged transitions, or uneven spacing between points, which can make later refinement steps less accurate.

## Node: ContourPointNoiseEyeballDistanceFilter

### What it does
ContourPointNoiseEyeballDistanceFilter removes contour points that are likely to be noise based on their distance from the eyeball boundary. By filtering out these unreliable points, the algorithm helps ensure that the remaining contour data more accurately represents the true shape of the iris and pupil boundaries.

### Key Parameters

| Parameter | Default | What it controls |
|---|---|---|
| min_distance_to_noise_and_eyeball | 0.005 | Minimum distance a contour point must be from the eyeball boundary or noise regions. Points closer than this threshold are removed to reduce noise in the contour data. |

### Code Example

The following code initializes the ContourPointNoiseEyeballDistanceFilter node. This node removes contour points that are too close to noise regions or the eyeball boundary, helping clean up the contour data.

In [ ]:
from iris.nodes.geometry_refinement.contour_points_filter import ContourPointNoiseEyeballDistanceFilter

noise_filter_node = ContourPointNoiseEyeballDistanceFilter()
print(noise_filter_node)  

### What to Look For

A successful run of the ContourPointNoiseEyeballDistanceFilter node should remove contour points that are too close to noise regions or the eyeball boundary.

Good output will keep the main contour shape while removing stray or unreliable points. Poor output may either leave too many noisy points in place or remove too many points, making the contour incomplete or distorted.

## Node: Smoothing

### What it does
Smoothing refines contour boundaries by reducing small irregularities and jaggedness in the contour points. This helps produce cleaner, more stable shapes for the pupil, iris, and eyeball before later stages of the pipeline use them.

### Key Parameters

| Parameter | Default | What it controls |
|---|---|---|
| dphi | 1.0 | Angular step used when sampling contour points during smoothing. Smaller values sample more points. |
| kernel_size | 10.0 | Size of the rolling median kernel used to smooth the contour boundary. |
| gap_threshold | 10.0 | Maximum allowed gap between contour points before smoothing adjusts the boundary. |

### Code Example

The following code initializes the Smoothing node. This node smooths contour boundaries by reducing jagged edges and small irregularities in the contour points.

In [ ]:
from iris.nodes.geometry_refinement.smoothing import Smoothing

smoothing_node = Smoothing()
print(smoothing_node)

### What to Look For

A successful run of the Smoothing node should produce contour boundaries that appear smoother and less jagged than the raw contours. Small irregularities and noise in the contour points should be reduced while still preserving the overall shape of the pupil, iris, and eyeball boundaries.

Good output will show smooth, continuous curves that follow the expected eye geometry. Poor output may still contain sharp angles, uneven spacing between points, or overly smoothed contours that distort the true boundary shape.

## Example Workflow: Visualizing Vectorization

The vectorization stage works on segmentation results rather than directly on the raw image. 
In a full pipeline run, segmentation first identifies the pupil, iris, and eyeball regions, and 
the vectorization nodes then convert those regions into contour-based representations.

The example below is included to help visualize how contour extraction and refinement fit into 
the Open-Iris pipeline.

In [ ]:
# Download a sample IR image for experimenting
!wget https://wld-ml-ai-data-public.s3.amazonaws.com/public-iris-images/example_orb_image_1.png -O ./sample_ir_image.png

In [ ]:
import cv2
import matplotlib.pyplot as plt

img_pixels = cv2.imread("./sample_ir_image.png", cv2.IMREAD_GRAYSCALE)

plt.figure(figsize=(5,5))
plt.imshow(img_pixels, cmap='gray')
plt.title("Sample IR Iris Image")
plt.axis("off")

In [ ]:
edges = cv2.Canny(img_pixels, 100, 200)

plt.figure(figsize=(5,5))
plt.imshow(edges, cmap="gray")
plt.title("Boundary Approximation")
plt.axis("off")

### Interpretation

The vectorization stage converts segmentation masks into contour-based representations of the pupil, iris, and eyeball boundaries. The edge visualization above illustrates the type of boundary information that later nodes refine using interpolation, filtering, and smoothing.